<a href="https://www.kaggle.com/code/michelepulvirenti/houseplant-recognition?scriptVersionId=267837511" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import os
import random
from tqdm import tqdm
import numpy as np
from PIL import Image
import math

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Parameters

In [ ]:
batch_size = 32
image_size = 256
dataset_dir = '/kaggle/input/house-plant-species/house_plant_species'

# Dataset loading

In [ ]:
# --- Image preprocessing ---
def preprocess_pil_image(pil_img):
    # Handle images with palette or transparency
    if pil_img.mode in ("P", "RGBA"):
        pil_img = pil_img.convert("RGBA")  # Preserve transparency if needed
        # Convert RGBA → RGB by compositing on white background
        background = Image.new("RGBA", pil_img.size, (255, 255, 255, 255))
        pil_img = Image.alpha_composite(background, pil_img).convert("RGB")
    elif pil_img.mode != "RGB":
        pil_img = pil_img.convert("RGB")

    img_resized = pil_img.resize((image_size, image_size))
    img_array = np.array(img_resized)
    return img_array



# --- Load dataset directly from Kaggle directory ---
class_names = sorted(os.listdir(dataset_dir))
class_to_id = {name: i for i, name in enumerate(class_names)}

print("🌿 Found classes:", class_names)

images, labels = [], []

# Loop through folders and images
for class_name in tqdm(class_names, desc="Preprocessing data"):
    class_path = os.path.join(dataset_dir, class_name)
    if not os.path.isdir(class_path):
        continue

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

        try:
            pil_image = Image.open(img_path)
            img_pp = preprocess_pil_image(pil_image)
            images.append(img_pp)
            labels.append(class_to_id[class_name])
        except Exception as e:
            print(f"⚠️ Failed to process image {img_path}: {e}")

images = np.array(images)
labels = np.array(labels)

print(f"✅ Preprocessed and saved dataset.\nImages shape: {images.shape}, labels shape: {labels.shape}")

# Dataset info

In [ ]:
unique_labels, counts = np.unique(labels, return_counts=True)
print("Labels distribution:", dict(zip(unique_labels, counts)))

num_classes = len(set(labels))
print("Number of classes:", num_classes)

In [ ]:
# print("Displaying one sample per label:")
# fig, axes = plt.subplots(1, len(unique_labels), figsize=(4 * len(unique_labels), 4))

# if len(unique_labels) == 1:
#     axes = [axes]  # Handle single-class edge case

# for i, label_id in enumerate(unique_labels):
#     # Find the first sample index for this label
#     idx = np.where(labels == label_id)[0][0]
#     img = images[idx]

#     axes[i].imshow(img)
#     axes[i].set_title(class_names[label_id])
#     axes[i].axis("off")

# plt.tight_layout()
# plt.show()

# Training

In [ ]:
labels_cat = to_categorical(labels, num_classes=num_classes)

# Step 3: Train-test split for validation
X_train, X_val, y_train, y_val = train_test_split(images, labels_cat, test_size=0.2, random_state=42, stratify=labels)

## Model

In [ ]:
# Step 4: Build the model (Transfer Learning)

base_model = EfficientNetV2B0(
    input_shape=(image_size, image_size, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)

base_model.trainable = False  # Freeze base model

inputs = layers.Input(shape=(image_size, image_size, 3))
x = base_model(inputs, training=False)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = models.Model(inputs, outputs)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=loss,
              metrics=['accuracy'])

model.summary()

## Data augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),

    layers.RandomZoom(0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),

    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.15),
    layers.RandomSaturation(0.15),

    layers.RandomHue(0.05),

    layers.GaussianNoise(0.02)
])


# Using augmentation in a dataset pipeline
def augment_and_preprocess(img):
    img = data_augmentation(tf.cast(img, tf.float32))
    img = preprocess_input(img)
    return img

# If you prepare a tf.data.Dataset (recommended for large datasets):
# Assuming X_train, y_train are NumPy arrays of images and labels
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(buffer_size=1024)
train_dataset = train_dataset.map(
    lambda x, y: (augment_and_preprocess(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# For validation, do NOT apply augmentation:
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
# sample_images, sample_labels = next(iter(train_dataset.unbatch().batch(batch_size)))

# plt.figure(figsize=(10, 10))

# num_samples = len(sample_images)
# cols = 4
# rows = math.ceil(num_samples / cols)

# for i in range(num_samples):
#     plt.subplot(rows, cols, i + 1)
#     img = np.clip(sample_images[i].numpy() / 255.0, 0, 1)
#     plt.imshow(img)
#     plt.title(f"Label: {np.argmax(sample_labels[i])}")
#     plt.axis("off")

# plt.suptitle("Sample Training Images (with Augmentation)", fontsize=14)
# plt.tight_layout()
# plt.show()


## Training model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# When training, use train_dataset and val_dataset instead of NumPy arrays
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    callbacks=[early_stop]
)

In [ ]:
# Extract metrics
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc) + 1)

# Plot Accuracy
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, 'b-', label='Training Accuracy')
plt.plot(epochs, val_acc, 'r-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, 'b-', label='Training Loss')
plt.plot(epochs, val_loss, 'r-', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
print(f"Training Accuracy: {acc[-1]:.4f}")
print(f"Validation Accuracy: {val_acc[-1]:.4f}")
print(f"Training Loss: {loss[-1]:.4f}")
print(f"Validation Loss: {val_loss[-1]:.4f}")

In [ ]:
# Step 6: Fine-tune the base model by unfreezing some layers
base_model.trainable = True
# Unfreeze only the top layers of base model for fine-tuning
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history_fine = model.fit(train_dataset,
                         validation_data=val_dataset,
                         epochs=50,
                         callbacks=[early_stop])

In [ ]:
# Extract metrics
acc_fine = history_fine.history['accuracy']
val_acc_fine = history_fine.history['val_accuracy']
loss_fine = history_fine.history['loss']
val_loss_fine = history_fine.history['val_loss']

epochs = range(1, len(acc_fine) + 1)

# Plot Accuracy
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, acc_fine, 'b-', label='Training Accuracy')
plt.plot(epochs, val_acc_fine, 'r-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, loss_fine, 'b-', label='Training Loss')
plt.plot(epochs, val_loss_fine, 'r-', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
print(f"Final Training Accuracy: {acc_fine[-1]:.4f}")
print(f"Final Validation Accuracy: {val_acc_fine[-1]:.4f}")
print(f"Final Training Loss: {loss_fine[-1]:.4f}")
print(f"Final Validation Loss: {val_loss_fine[-1]:.4f}")

# Saving model

In [ ]:
# Step 7: Export model to a TensorFlow SavedModel directory
MODEL_DIR = '/kaggle/working/houseplant_classifier_model'  # no .keras extension
model.export(MODEL_DIR)  # creates a folder with saved_model.pb etc.

# Step 8: Convert to TensorFlow Lite model
converter = tf.lite.TFLiteConverter.from_saved_model(MODEL_DIR)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('/kaggle/working/houseplant_classifier_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("✅ TensorFlow Lite model saved as houseplant_classifier_model.tflite")